# 03. SQL Insert, Update, Delete & DML: Beginner Guide

### 📝 Universal SQL Execution Order (All SQL Clauses Combined):
```text
┌─ Complete All-in-One SQL Logical Execution Pipeline (All 12 Clauses) ────────┐
│ 1. WITH (CTEs)       ➔ 2. FROM & JOIN (ON)   ➔ 3. WHERE (Row Filter)         │
│ ➔ 4. GROUP BY        ➔ 5. HAVING (Agg Filter)➔ 6. WINDOW (OVER / Partition)  │
│ ➔ 7. QUALIFY         ➔ 8. SELECT & CASE      ➔ 9. DISTINCT (Dedup)           │
│ ➔ 10. UNION/INTERSECT➔ 11. ORDER BY (Sort)   ➔ 12. LIMIT / OFFSET (Page)     │
└──────────────────────────────────────────────────────────────────────────────┘
```

---

### 📌 Overview & Architectural Context
Welcome to **03. SQL Insert, Update, Delete & DML**. While DDL defines structural table blueprints, DML commands mutate the active row states stored within those structures. This notebook covers record creation (`INSERT INTO`), selective state updates (`UPDATE`), transactional row deletions (`DELETE`), full storage resets (`TRUNCATE`), and modern UPSERT / MERGE strategies (`INSERT ... ON CONFLICT DO UPDATE`).

### 📚 Key Concepts Covered in this Notebook:
- [x] 🔹 Single & Bulk Row Insertion: `INSERT INTO ... VALUES`
- [x] 🔹 Insert from Query Projection: `INSERT INTO ... SELECT`
- [x] 🔹 Conditional State Updates: `UPDATE ... SET ... WHERE`
- [x] 🔹 Transactional Row Deletion: `DELETE FROM ... WHERE`
- [x] 🔹 Modern Upsert Logic: `INSERT ... ON CONFLICT DO UPDATE`
- [x] 🔍 Scenario: Processing Real-Time Chargeback Dispute Status Updates











In [1]:
# Setup in-memory SQLite relational engine with Native SQL Studio Execution
import sqlite3
import pandas as pd
import os
from IPython import get_ipython
from IPython.core.magic import register_line_cell_magic

conn = sqlite3.connect(':memory:')

def load_table(name, path):
    if os.path.exists(path):
        df = pd.read_csv(path)
        df.to_sql(name, conn, index=False, if_exists='replace')

load_table('transactions', 'data/raw_transactions.csv' if os.path.exists('data/raw_transactions.csv') else '../data/raw_transactions.csv')
load_table('customers', 'data/customers.csv' if os.path.exists('data/customers.csv') else '../data/customers.csv')
load_table('merchants', 'data/merchants.csv' if os.path.exists('data/merchants.csv') else '../data/merchants.csv')
load_table('disputes', 'data/disputes.csv' if os.path.exists('data/disputes.csv') else '../data/disputes.csv')

def _execute_raw_sql(query):
    query = query.strip()
    if query.upper().startswith(('INSERT', 'UPDATE', 'DELETE', 'CREATE', 'DROP', 'ALTER', 'VACUUM', 'ANALYZE', 'BEGIN', 'COMMIT', 'ROLLBACK', 'SAVEPOINT')):
        cur = conn.cursor()
        cur.executescript(query)
        conn.commit()
        return "Query Executed Successfully."
    else:
        return pd.read_sql_query(query, conn)

# Register automatic raw SQL transformer & %%sql magic
ip = get_ipython()
if ip is not None:
    def raw_sql_transformer(lines):
        clean_text = ''.join(lines).strip()
        first_token = clean_text.split()[0].upper() if clean_text.split() else ''
        sql_keywords = {'SELECT', 'WITH', 'INSERT', 'UPDATE', 'DELETE', 'CREATE', 'DROP', 'ALTER', 'EXPLAIN', 'ANALYZE', 'VACUUM', 'BEGIN', 'COMMIT', 'ROLLBACK'}
        if first_token in sql_keywords:
            return [f'_execute_raw_sql("""{clean_text}""")']
        return lines
    
    if raw_sql_transformer not in ip.input_transformers_cleanup:
        ip.input_transformers_cleanup.append(raw_sql_transformer)

@register_line_cell_magic
def sql(line, cell=None):
    return _execute_raw_sql(cell if cell is not None else line)

print("SQL Studio Environment Active! You can now write and run pure SQL queries directly.")


SQL Studio Environment Active! You can now write and run pure SQL queries directly.


### 🔹 Row Insertion: `INSERT INTO ... VALUES`
- **What it does:** Appends new tuples into a target table, validating column constraints and auto-populating default values.
- **Syntax:** `INSERT INTO table_name (col1, col2) VALUES (val1, val2), (val3, val4)`
- **Dataset Application & Code Demonstration:** Inserts new audit dispute records into a temporary audit table.


In [2]:
%%sql
CREATE TABLE IF NOT EXISTS dispute_audit (
    dispute_id TEXT PRIMARY KEY,
    transaction_id TEXT,
    dispute_amount REAL,
    status TEXT DEFAULT 'PENDING'
);
INSERT INTO dispute_audit (dispute_id, transaction_id, dispute_amount, status)
VALUES 
    ('DSP_9001', 'TX_1001', 450.00, 'OPEN'),
    ('DSP_9002', 'TX_1002', 120.50, 'UNDER_REVIEW');
SELECT * FROM dispute_audit;


'Query Executed Successfully.'

### 🔹 Bulk Insertion from Query: `INSERT INTO ... SELECT`
- **What it does:** Populates a target table dynamically from the projected result set of a source query.
- **Syntax:** `INSERT INTO target_table (cols...) SELECT cols... FROM source_table WHERE condition`
- **Dataset Application & Code Demonstration:** Populates the audit table with high-risk fraudulent transactions.


In [3]:
%%sql
INSERT INTO dispute_audit (dispute_id, transaction_id, dispute_amount, status)
SELECT 
    'AUTO_' || transaction_id AS dispute_id,
    transaction_id,
    transaction_amount AS dispute_amount,
    'FLAGGED_FRAUD' AS status
FROM transactions
WHERE is_fraud = 1 AND transaction_amount > 1800.00
LIMIT 3;
SELECT * FROM dispute_audit;


'Query Executed Successfully.'

### 🔹 Conditional Updates: `UPDATE ... SET ... WHERE`
- **What it does:** Mutates attribute values in existing rows matching a predicate filter.
- **Syntax:** `UPDATE table_name SET col1 = val1, col2 = expr WHERE condition`
- **Dataset Application & Code Demonstration:** Resolves and updates pending disputes.


In [4]:
%%sql
UPDATE dispute_audit
SET status = 'RESOLVED'
WHERE dispute_id = 'DSP_9001';
SELECT * FROM dispute_audit WHERE dispute_id = 'DSP_9001';


'Query Executed Successfully.'

### 🔹 Row Deletion: `DELETE FROM ... WHERE`
- **What it does:** Removes specific rows matching a predicate while logging each deleted row in the write-ahead transaction log.
- **Syntax:** `DELETE FROM table_name WHERE condition`
- **Dataset Application & Code Demonstration:** Removes resolved disputes from the active working queue.


In [5]:
%%sql
DELETE FROM dispute_audit
WHERE status = 'RESOLVED';
SELECT * FROM dispute_audit;


'Query Executed Successfully.'

### 🔹 Upsert Mechanics: `INSERT ... ON CONFLICT DO UPDATE`
- **What it does:** Atomically inserts a new row or updates existing attributes if a primary key or unique constraint violation occurs.
- **Syntax:** `INSERT INTO table (pk, val) VALUES (k, v) ON CONFLICT(pk) DO UPDATE SET val = excluded.val`
- **Dataset Application & Code Demonstration:** Upserts a dispute record by updating its amount if it already exists.


In [6]:
%%sql
INSERT INTO dispute_audit (dispute_id, transaction_id, dispute_amount, status)
VALUES ('DSP_9002', 'TX_1002', 300.00, 'ESCALATED')
ON CONFLICT(dispute_id) DO UPDATE SET
    dispute_amount = excluded.dispute_amount,
    status = excluded.status;
SELECT * FROM dispute_audit WHERE dispute_id = 'DSP_9002';


'Query Executed Successfully.'

## 💡 Real-World Practice & Scenarios
Practical scenarios and common data engineering questions explained with real examples.


### 🔍 Scenario: Q1: DELETE vs TRUNCATE vs DROP Storage Internals
- **Objective:** Compare transactional row deletion (`DELETE`), metadata page deallocation (`TRUNCATE`), and schema destruction (`DROP`).
- **Approach:** Analyze transaction log footprint, rollback capability, and table reset mechanics.


In [7]:
%%sql
SELECT 
    'DELETE' AS operation, 'Row-by-Row' AS granularity, 'Yes' AS can_rollback, 'High' AS wal_logging
UNION ALL
SELECT 'TRUNCATE', 'Page Deallocation', 'Yes (Engine dependent)', 'Minimal'
UNION ALL
SELECT 'DROP', 'Schema & File Removal', 'Yes (Within DDL Tx)', 'Minimal';


,operation,granularity,can_rollback,wal_logging
0,DELETE,Row-by-Row,Yes,High
1,TRUNCATE,Page Deallocation,Yes (Engine dependent),Minimal
2,DROP,Schema & File Removal,Yes (Within DDL Tx),Minimal
